# Dropout Rate to Noise Std Mapping

Build the equivalence mapping between `dropout_rate` and `noise_std`
using the localization sweep data. See `MAP_PLAN.md` for the full plan.

In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt

# Ensure notebooks/ is on the import path regardless of kernel cwd
_nb_dir = (
    pathlib.Path(__file__).resolve().parent
    if "__file__" in dir()
    else pathlib.Path.cwd()
)
if _nb_dir.name != "notebooks":
    _nb_dir = _nb_dir / "notebooks"
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

_repo_root = _nb_dir.parent

import equivalence_map as em  # noqa: E402
from wandb_cache import load_sweep  # noqa: E402

In [ ]:
PROJECT = "llm-mechanistic-detection"
DROPOUT_SWEEP = "v3b0x2dq"
NOISE_SWEEP = "3op4712y"

# Dropout rates to exclude (artifact: near-perfect accuracy at 99%)
EXCLUDE_DROPOUT = [0.99]

## Step 1: Audit and aggregate the data

Load the sweep summaries, check for duplicates and missing conditions,
then aggregate into clean tables.

In [ ]:
# Load sweep summaries (cached locally after first download)
df_dropout_raw = load_sweep(DROPOUT_SWEEP, project=PROJECT)
df_noise_raw = load_sweep(NOISE_SWEEP, project=PROJECT)

print(f"Dropout sweep: {len(df_dropout_raw)} runs")
print(f"Noise sweep:   {len(df_noise_raw)} runs")

In [ ]:
# Audit both sweeps
print("=== Dropout sweep audit ===")
audit_d = em.audit_sweep(df_dropout_raw, em.COL_DROPOUT)
print(f"  Runs: {audit_d['n_runs']}")
print(f"  Unique grid cells: {audit_d['n_unique_cells']}")
print(f"  Duplicates: {audit_d['n_duplicates']}")
print(f"  Missing cells: {audit_d['n_missing']}")
print(f"  Perturbation values: {audit_d['n_perturbation_values']}")

if audit_d["duplicates"] is not None:
    print("\n  Duplicate conditions:")
    print(audit_d["duplicates"].to_string(index=False))

if audit_d["missing"] is not None:
    print("\n  Missing conditions (showing first 10):")
    print(audit_d["missing"].head(10).to_string(index=False))

print("\n=== Noise sweep audit ===")
audit_n = em.audit_sweep(df_noise_raw, em.COL_NOISE)
print(f"  Runs: {audit_n['n_runs']}")
print(f"  Unique grid cells: {audit_n['n_unique_cells']}")
print(f"  Duplicates: {audit_n['n_duplicates']}")
print(f"  Missing cells: {audit_n['n_missing']}")
print(f"  Perturbation values: {audit_n['n_perturbation_values']}")

if audit_n["duplicates"] is not None:
    print("\n  Duplicate conditions:")
    print(audit_n["duplicates"].to_string(index=False))

if audit_n["missing"] is not None:
    print("\n  Missing conditions (showing first 10):")
    print(audit_n["missing"].head(10).to_string(index=False))

In [ ]:
# Aggregate: one row per (condition, perturbation_value), collapsing duplicates
df_dropout_agg = em.aggregate_sweep(
    df_dropout_raw,
    em.COL_DROPOUT,
    exclude_perturbation=EXCLUDE_DROPOUT,
)
df_noise_agg = em.aggregate_sweep(df_noise_raw, em.COL_NOISE)

print(
    f"Dropout aggregated: {len(df_dropout_agg)} rows "
    f"(excluded dropout_rate in {EXCLUDE_DROPOUT})"
)
print(f"Noise aggregated:   {len(df_noise_agg)} rows")
df_dropout_agg.head()

## Step 2: Plot raw curves

Visualize performance vs perturbation strength for every condition.
Check: curve shape, monotonicity, location of steep transitions, peak behavior.

In [ ]:
# Add sentence length labels for cleaner plots
df_dropout_agg["sentence_label"] = df_dropout_agg[em.COL_SENTENCES].apply(
    em.sentence_label
)
df_noise_agg["sentence_label"] = df_noise_agg[em.COL_SENTENCES].apply(em.sentence_label)

models = sorted(df_dropout_agg[em.COL_MODEL].unique())
prompts = sorted(df_dropout_agg[em.COL_PROMPT_TURNS].unique())
tok_labels = sorted(
    df_dropout_agg["sentence_label"].unique(),
    key=lambda s: int(s.replace("tok", "")),
)

print(f"Models:  {models}")
print(f"Prompts: {prompts}")
print(f"Tokens:  {tok_labels}")

In [ ]:
# Plot raw curves: one figure per model, columns = sentence lengths,
# rows = dropout / noise, colors = prompt templates.
# sharey=True across both rows so dropout and noise are comparable.

for model in models:
    fig, axes = plt.subplots(
        2,
        len(tok_labels),
        figsize=(4 * len(tok_labels), 7),
        sharey=True,
    )
    fig.suptitle(model, fontsize=14, fontweight="bold")

    for j, tok in enumerate(tok_labels):
        # Dropout row
        ax_d = axes[0, j] if len(tok_labels) > 1 else axes[0]
        for prompt in prompts:
            mask = (
                (df_dropout_agg[em.COL_MODEL] == model)
                & (df_dropout_agg[em.COL_PROMPT_TURNS] == prompt)
                & (df_dropout_agg["sentence_label"] == tok)
            )
            sub = df_dropout_agg[mask].sort_values(em.COL_DROPOUT)
            if len(sub) == 0:
                continue
            label = prompt.replace("localization_", "").replace("_2_letters", "")
            ax_d.plot(
                sub[em.COL_DROPOUT], sub["metric_mean"], ".-", label=label, markersize=3
            )
            ax_d.fill_between(
                sub[em.COL_DROPOUT],
                sub["metric_mean"] - sub["metric_se"],
                sub["metric_mean"] + sub["metric_se"],
                alpha=0.15,
            )
        ax_d.set_title(tok)
        if j == 0:
            ax_d.set_ylabel("log P(correct) — dropout")

        # Noise row
        ax_n = axes[1, j] if len(tok_labels) > 1 else axes[1]
        for prompt in prompts:
            mask = (
                (df_noise_agg[em.COL_MODEL] == model)
                & (df_noise_agg[em.COL_PROMPT_TURNS] == prompt)
                & (df_noise_agg["sentence_label"] == tok)
            )
            sub = df_noise_agg[mask].sort_values(em.COL_NOISE)
            if len(sub) == 0:
                continue
            label = prompt.replace("localization_", "").replace("_2_letters", "")
            ax_n.plot(
                sub[em.COL_NOISE], sub["metric_mean"], ".-", label=label, markersize=3
            )
            ax_n.fill_between(
                sub[em.COL_NOISE],
                sub["metric_mean"] - sub["metric_se"],
                sub["metric_mean"] + sub["metric_se"],
                alpha=0.15,
            )
        ax_n.set_xlabel("noise_std")
        if j == 0:
            ax_n.set_ylabel("log P(correct) — noise")

    # Add legend to last column
    axes[0, -1].legend(fontsize=7, loc="lower right")
    axes[1, -1].legend(fontsize=7, loc="lower right")

    # Add x-label to first row
    for j in range(len(tok_labels)):
        ax_d = axes[0, j] if len(tok_labels) > 1 else axes[0]
        ax_d.set_xlabel("dropout_rate")

    plt.tight_layout()
    plt.show()

## Steps 3-5: Monotonic range, isotonic regression, inversion

For each condition: find the monotonic range, fit isotonic regression,
invert the noise curve to find matched pairs.

In [ ]:
# Build mappings for all conditions
all_mappings, details = em.build_all_mappings(
    df_dropout_agg,
    df_noise_agg,
    exclude_dropout=EXCLUDE_DROPOUT,
)

n_conditions = len(details)
n_valid = all_mappings["in_valid_range"].sum()
n_total = len(all_mappings)

print(f"Conditions processed: {n_conditions}")
print(
    f"Matched pairs: {n_valid} / {n_total} "
    f"({100 * n_valid / n_total:.1f}% in valid range)"
)
all_mappings.head(10)

## Step 6a: Inspect the isotonic fits

For a few representative conditions, plot the raw data, isotonic fit,
and the monotonic range boundaries. This verifies the pipeline behaves well.

In [ ]:
# Show isotonic fits for up to 6 conditions (one per model x first prompt)
sample_conditions = [c for c in sorted(details.keys()) if c[1] == prompts[0]][:6]

for condition in sample_conditions:
    info = details[condition]
    label = em.condition_label(condition)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(label, fontsize=11)

    # -- Dropout: raw + isotonic --
    ax = axes[0]
    d_mask = (
        (df_dropout_agg[em.COL_MODEL] == condition[0])
        & (df_dropout_agg[em.COL_PROMPT_TURNS] == condition[1])
        & (df_dropout_agg[em.COL_SENTENCES] == condition[2])
    )
    d_raw = df_dropout_agg[d_mask].sort_values(em.COL_DROPOUT)
    ax.plot(
        d_raw[em.COL_DROPOUT],
        d_raw["metric_mean"],
        "o",
        markersize=3,
        label="raw",
        alpha=0.5,
    )

    d_iso_vals, d_iso_perf = info["dropout_isotonic"]
    ax.plot(d_iso_vals, d_iso_perf, "-", linewidth=2, label="isotonic")

    d_lo, d_hi = info["dropout_range"]
    ax.axvline(
        d_raw[em.COL_DROPOUT].iloc[d_lo] if d_lo < len(d_raw) else 0,
        color="gray",
        linestyle="--",
        alpha=0.5,
        label="range bounds",
    )
    ax.axvline(
        d_raw[em.COL_DROPOUT].iloc[min(d_hi, len(d_raw) - 1)],
        color="gray",
        linestyle="--",
        alpha=0.5,
    )
    ax.set_xlabel("dropout_rate")
    ax.set_ylabel("log P(correct)")
    ax.set_title("Dropout")
    ax.legend(fontsize=7)

    # -- Noise: raw + isotonic --
    ax = axes[1]
    n_mask = (
        (df_noise_agg[em.COL_MODEL] == condition[0])
        & (df_noise_agg[em.COL_PROMPT_TURNS] == condition[1])
        & (df_noise_agg[em.COL_SENTENCES] == condition[2])
    )
    n_raw = df_noise_agg[n_mask].sort_values(em.COL_NOISE)
    ax.plot(
        n_raw[em.COL_NOISE],
        n_raw["metric_mean"],
        "o",
        markersize=3,
        label="raw",
        alpha=0.5,
    )

    n_iso_vals, n_iso_perf = info["noise_isotonic"]
    ax.plot(n_iso_vals, n_iso_perf, "-", linewidth=2, label="isotonic")

    n_lo, n_hi = info["noise_range"]
    ax.axvline(
        n_raw[em.COL_NOISE].iloc[n_lo] if n_lo < len(n_raw) else 0,
        color="gray",
        linestyle="--",
        alpha=0.5,
        label="range bounds",
    )
    ax.axvline(
        n_raw[em.COL_NOISE].iloc[min(n_hi, len(n_raw) - 1)],
        color="gray",
        linestyle="--",
        alpha=0.5,
    )
    ax.set_xlabel("noise_std")
    ax.set_title("Noise")
    ax.legend(fontsize=7)

    # -- Mapping --
    ax = axes[2]
    mapping = info["mapping"]
    valid = mapping[mapping["in_valid_range"]]
    ax.plot(valid["dropout_rate"], valid["matched_noise_std"], "o-", markersize=4)
    ax.set_xlabel("dropout_rate")
    ax.set_ylabel("matched noise_std")
    ax.set_title(f"Mapping ({len(valid)} valid pairs)")

    plt.tight_layout()
    plt.show()

## Step 6b: Mapping overview

Plot all condition mappings together, faceted by model.

In [ ]:
# Add sentence label to mappings for plotting
all_mappings["sentence_label"] = all_mappings[em.COL_SENTENCES].apply(em.sentence_label)
valid_mappings = all_mappings[all_mappings["in_valid_range"]].copy()

fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 5), sharey=True)
if len(models) == 1:
    axes = [axes]

for i, model in enumerate(models):
    ax = axes[i]
    sub = valid_mappings[valid_mappings[em.COL_MODEL] == model]

    for tok in tok_labels:
        for prompt in prompts:
            mask = (sub["sentence_label"] == tok) & (sub[em.COL_PROMPT_TURNS] == prompt)
            s = sub[mask].sort_values("dropout_rate")
            if len(s) == 0:
                continue
            prompt_short = prompt.replace("localization_", "").replace("_2_letters", "")
            ax.plot(
                s["dropout_rate"],
                s["matched_noise_std"],
                ".-",
                markersize=4,
                label=f"{tok} {prompt_short}",
            )

    ax.set_xlabel("dropout_rate")
    ax.set_title(model)
    if i == 0:
        ax.set_ylabel("matched noise_std")
    ax.legend(fontsize=6, loc="upper left", ncol=2)

fig.suptitle("Dropout → Noise Mapping (all conditions)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Step 7: Universality analysis

Do the mapping curves collapse onto a single curve, or do they depend
on model, prompt, or sentence length?

In [ ]:
# Overlay all mapping curves on one plot, colored by model
fig, ax = plt.subplots(figsize=(8, 6))

colors_model = {m: c for m, c in zip(models, plt.cm.tab10.colors)}
style_prompt = {p: s for p, s in zip(prompts, ["-", "--"])}

for model in models:
    for prompt in prompts:
        for tok in tok_labels:
            mask = (
                (valid_mappings[em.COL_MODEL] == model)
                & (valid_mappings[em.COL_PROMPT_TURNS] == prompt)
                & (valid_mappings["sentence_label"] == tok)
            )
            s = valid_mappings[mask].sort_values("dropout_rate")
            if len(s) == 0:
                continue
            ax.plot(
                s["dropout_rate"],
                s["matched_noise_std"],
                style_prompt[prompt],
                color=colors_model[model],
                alpha=0.5,
                linewidth=1,
                markersize=2,
            )

# Legend entries (model colors + prompt styles)
for model in models:
    ax.plot([], [], "-", color=colors_model[model], label=model)
for prompt in prompts:
    prompt_short = prompt.replace("localization_", "").replace("_2_letters", "")
    ax.plot([], [], style_prompt[prompt], color="gray", label=prompt_short)

ax.set_xlabel("dropout_rate")
ax.set_ylabel("matched noise_std")
ax.set_title("Universality: all conditions overlaid")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Quantify spread: for each dropout_rate, compute std of matched noise_std
# across all conditions
spread = (
    valid_mappings.groupby("dropout_rate")["matched_noise_std"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
spread = spread[spread["count"] >= 3]  # need at least 3 conditions

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.errorbar(
    spread["dropout_rate"],
    spread["mean"],
    yerr=spread["std"],
    fmt="o-",
    markersize=3,
    capsize=2,
)
ax.set_xlabel("dropout_rate")
ax.set_ylabel("matched noise_std (mean ± std)")
ax.set_title("Cross-condition mean mapping")

ax = axes[1]
ax.plot(spread["dropout_rate"], spread["std"], "o-", markersize=3)
ax.set_xlabel("dropout_rate")
ax.set_ylabel("std of matched noise_std")
ax.set_title("Cross-condition spread")

plt.tight_layout()
plt.show()

print(f"\nMedian cross-condition std: {spread['std'].median():.4f}")
print(f"Mean cross-condition std:   {spread['std'].mean():.4f}")

## Export

Save the matched pairs to CSV for use in downstream experiments.

In [ ]:
output_dir = _repo_root / "data" / "equivalence_map"
output_dir.mkdir(parents=True, exist_ok=True)

# Per-condition mapping (with dropout_se column)
output_path = output_dir / "matched_pairs.csv"
valid_mappings.to_csv(output_path, index=False)
print(f"Saved {len(valid_mappings)} matched pairs to {output_path}")

# Cross-condition mean mapping
mean_path = output_dir / "mean_mapping.csv"
spread.to_csv(mean_path, index=False)
print(f"Saved cross-condition mean mapping to {mean_path}")